# Irish Housing Density: Clustering Census 2022 Small Areas
### A CRISP-DM Clustering Project

**Student:** Patryk [Surname] | [Student number] | MTU Kerry
**Module:** Data Analytics, CA2 | **Lecturer:** Andrew Shields
**Repository:** [INSERT GITHUB URL]

This notebook follows the six phases of CRISP-DM (Cross-Industry Standard Process for Data Mining):

1. Business Understanding
2. Data Understanding
3. Data Preparation
4. Modelling
5. Evaluation
6. Deployment and Recommendations

It ends with a personal reflection and a Responsible AI Collaboration Statement.

## Setup

All libraries are imported here, and the random seed is fixed so that the results are the same every time the notebook is run (**Kernel → Restart & Run All**). Library versions are printed for reproducibility, and the code checks that the raw data files are in place.

In [ ]:
# Core libraries
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

# Fixed random seed so results are reproducible
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Relative paths (the notebook lives in notebooks/, the data in data/)
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
FIG_DIR = Path("../figures")

SAPS_FILE = RAW_DIR / "SAPS_2022_Small_Area_UR_171024.csv"
BOUNDARY_FILE = RAW_DIR / "Small_Area_National_Statistical_Boundaries_2022_Ungeneralised_view_-4936014253783534521.csv"
GLOSSARY_FILE = RAW_DIR / "Glossary_Saps_2022_REVISED_21102024.xlsx"

# Check that every raw file is where we expect it
for f in [SAPS_FILE, BOUNDARY_FILE, GLOSSARY_FILE]:
    print(("OK      " if f.exists() else "MISSING ") + f.name)

# Print versions so the environment can be reproduced
print()
print("Python      :", sys.version.split()[0])
print("pandas      :", pd.__version__)
print("numpy       :", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("seaborn     :", sns.__version__)

---
# Phase 1: Business Understanding

This phase defines the real problem before looking at the data or the method. It sets out who needs the result, what decision it supports and what would count as success.

## 1.1 Context and Problem

Ireland's population passed 5.1 million in Census 2022, up 7.6% since 2016, while the housing stock grew by only about 6% over the same period [4]. To catch up, the revised National Planning Framework (NPF, April 2025) sets a target of around 50,000 new homes a year up to 2040 [3].

The NPF also says *where* new homes should go. It calls for **compact growth**: at least 40% of all new homes nationally should be built within the existing built-up footprint of cities, towns and villages, rising to at least 50% in the five cities and at least 30% in other settlements [3]. Compact growth makes better use of land, and it makes public transport, water, roads and broadband cheaper to provide per home.

At the same time, much of rural Ireland is built at very low density, and a large share of rural homes are one-off houses: single houses on their own sites rather than in a town or estate [6]. Ireland therefore has a wide mix of housing patterns, from dense city apartments to scattered rural homes.

**The problem:** planners in county councils and national bodies must decide where to focus compact growth, but Ireland has nearly 19,000 small areas with very different housing patterns. There is no simple, data-based way to sort these areas into types and quickly spot which low-density areas are well placed for more housing, for example because they are near a town, have public transport or already have vacant homes.

## 1.2 Stakeholders

| Role | Who | Interest |
|---|---|---|
| **Data suppliers** | Central Statistics Office (CSO, census data); Tailte Éireann (boundaries and land area) | Correct, fair use of their open data |
| **Primary users** | County council planners; Department of Housing, Local Government and Heritage | Where to focus compact growth and new housing |
| **Secondary users** | Land Development Agency; regional assemblies; transport planners | Planning land use and services |
| **Affected people** | Residents of areas grouped as "suited to more housing" | *Benefit:* more homes and services nearby. *Risk:* change to their area without consultation, or a label that oversimplifies their community |

## 1.3 Business (Practical) Objective

Give planners an evidence-based **map of housing-pattern types** across Ireland's small areas, so they can quickly find areas that are **low-density but well placed for more housing** (near towns, served by public transport, or with high vacancy). Compact growth effort can then be targeted there, in line with the NPF.

The project shortlists *types* of area for planners to look at more closely. It does **not** choose specific building sites; people make those decisions.

## 1.4 Analytics Objective

**Task type:** clustering (unsupervised learning). There is no target column to predict; the goal is to find natural groups in the data.

**Unit of analysis:** one CSO Small Area (an area of usually 80–120 dwellings). There are 18,919 small areas.

**Method:** k-means clustering on eight features built from the census and boundary data:

1. Dwellings per km² (log scale)
2. % of households in flats/apartments
3. % of homes built since 2011
4. Vacancy rate
5. Holiday-home rate
6. Persons per household
7. % of households renting
8. % commuting to work by public transport

Each resulting cluster is then described in plain language, for example "dense city apartments, mostly rented" or "low-density town edge with bus links and vacant homes".

**Analytical question:**
> *Can Irish small areas be grouped into distinct housing-pattern types to help planners identify low-density areas that are well placed for additional housing, in line with the compact growth goals of the National Planning Framework?*

## 1.5 Success Criteria

The criteria are split into **technical** (is the clustering sound?) and **practical** (is it useful to planners?). These are first targets and will be revisited in Phase 5.

**Technical success criteria**

| ID | Criterion | Target |
|---|---|---|
| T1 | **Silhouette score** (how well separated the clusters are, from −1 to 1) | ≥ 0.25 for the chosen number of clusters |
| T2 | **Stability**: the same clusters appear when k-means is re-run with different random starts | Adjusted Rand Index ≥ 0.80 between runs |
| T3 | **Cluster sizes**: no cluster is too small to act on | Each cluster ≥ 1% of small areas (about 190), unless a smaller one is clearly meaningful |
| T4 | **Comparison with a reference grouping**: the clusters add information beyond the CSO's existing six urban/rural categories | At least one cluster is not just a copy of an existing CSO category |
| T5 | **Reproducibility** | Restart & Run All gives the same results (fixed seed) |

**Practical success criteria**

| ID | Criterion |
|---|---|
| P1 | Every cluster can be described in one plain sentence that a non-technical planner understands |
| P2 | At least one cluster picks out low-density areas with signs that they could take more housing (public transport, vacancy, closeness to towns) |
| P3 | Results can be summarised per county, the level at which councils plan |
| P4 | Results are presented as a shortlist for human review, not as automatic decisions |

**Why a reference grouping matters:** if the clusters only repeated the CSO's urban/rural labels, the model would add nothing new. The CSO categories act as the **baseline** that the clustering must improve on.

## 1.6 Constraints, Assumptions and Risks

**Constraints**
- **Time:** individual project, final submission Friday 4 December 2026.
- **Data:** open data only. The census is a snapshot of one night (3 April 2022), so vacancy is a point-in-time measure and includes short-term vacancy such as homes for sale or being renovated [5].
- **Missing information:** the data says nothing about zoning, land ownership, flood risk or water and wastewater capacity, which all decide whether land can actually be built on.

**Assumptions**
- Land area from the boundary file is a fair basis for density. Note that it is **gross density**: it includes fields, roads and water inside each area, not only residential land.
- Patterns from 2022 are still broadly relevant, since the next census is in 2027.

**Risks**

| Risk | Why it matters | Planned response |
|---|---|---|
| **Very uneven values** | Density ranges from near 0 to tens of thousands of dwellings per km², which would dominate k-means | Log-transform density; standardise all features (Phase 3) |
| **Different scales** | k-means uses distances, so a feature with big numbers would outweigh one with small numbers | Standardise every feature to mean 0 and standard deviation 1 |
| **Overlapping features** | Two features measuring the same thing would be counted twice | Check correlations; % car commuting already removed (r = −0.65 with public transport) |
| **Choosing the number of clusters** | There is no single correct k | Compare k values with the elbow method and silhouette scores, and check that each cluster makes sense |
| **Odd or special areas** | National total row; 3 areas with people but no households (likely communal buildings such as student residences) | Remove and record in the preparation log (Phase 3) |
| **Ecological fallacy** | A cluster describes an area as a whole, not every home or person in it | Only make claims about areas, never individuals |
| **Misuse** | Labels such as "low density" could be used to push development without consultation | Present results as a shortlist for planners, with local consultation and human judgement (Phase 6) |

**Ethics and privacy:** all data is aggregated open data published under CC BY 4.0. The CSO applies disclosure control to small areas, so no individual or household can be identified.

## 1.7 Initial Project Plan

| CRISP-DM phase | Main activities | Target week |
|---|---|---|
| 1. Business Understanding | Problem, stakeholders, objectives, success criteria, risks | Week 8 (3–9 Nov) |
| 2. Data Understanding | Collection, description, exploration and quality reports; charts of density and housing type by urban/rural category | Week 8–9 |
| 3. Data Preparation | Join census and boundary data on GUID; remove the national total and special areas; build the 8 features; log-transform and standardise; preparation log | Week 9 (10–16 Nov) |
| 4. Modelling | Elbow method and silhouette scores to choose k; k-means; one alternative for comparison (e.g. a different k or hierarchical clustering on a sample) | Week 10 (17–23 Nov) |
| 5. Evaluation | Silhouette, stability, cluster sizes, comparison with CSO urban/rural categories, cluster profiles, practical usefulness | Week 10–11 |
| 6. Deployment | Recommendations for planners, monitoring, limitations, reflection | Week 11 (24–30 Nov) |
| Paper and final checks | 4-page paper, AI statement, Restart & Run All, submit | Week 12 (1–4 Dec) |

**Why k-means?** It is simple, fast on 18,919 rows and easy to explain: each area joins the group whose average profile it is closest to. Each cluster's average profile can be read straight off as a description of that housing type.

### Phase 1 References

[3] Government of Ireland, "National Planning Framework: First Revision," Apr. 2025. [Online]. Available: https://cdn.npf.ie/wp-content/uploads/National-Planning-Framework-First-Revision-April-2025.pdf [Accessed: 03-Nov-2026].

[4] Central Statistics Office, "Census of Population 2022 – Preliminary Results: Introduction," 2022. [Online]. Available: https://www.cso.ie/en/releasesandpublications/ep/p-cpr/censusofpopulation2022-preliminaryresults/introduction/ [Accessed: 03-Nov-2026].

[5] Central Statistics Office, "Census 2022 Profile 2 – Housing in Ireland: Key Findings," Jul. 2023. [Online]. Available: https://www.cso.ie/en/releasesandpublications/ep/p-cpp2/censusofpopulation2022profile2-housinginireland/keyfindings/ [Accessed: 03-Nov-2026].

[6] Central Statistics Office, "CD457 – Percentage of One Off Housing in Rural Areas," data.gov.ie. [Online]. Available: https://data.gov.ie/dataset/cd457-percentage-of-one-off-housing-in-rural-areas [Accessed: 03-Nov-2026].

Numbering follows `references/references.md`, where [1] and [2] are the two datasets (cited in Phase 2).

---
# Phase 2: Data Understanding

*To be completed.*

## 2.1 Initial Data Collection Report

*To be completed.*

## 2.2 Data Description Report

*To be completed.*

## 2.3 Data Exploration Report

*To be completed.*

## 2.4 Data Quality Report

*To be completed.*

---
# Phase 3: Data Preparation

*To be completed.*

## 3.1 Selection

*To be completed.*

## 3.2 Cleaning

*To be completed.*

## 3.3 Integration (Join on GUID)

*To be completed.*

## 3.4 Feature Construction

*To be completed.*

## 3.5 Transformation and Scaling

*To be completed.*

## 3.6 Embedded Preparation Log

*To be completed.*

## 3.7 Validation (Before/After)

*To be completed.*

---
# Phase 4: Modelling

*To be completed.*

## 4.1 Choosing k (Elbow and Silhouette)

*To be completed.*

## 4.2 K-Means Model

*To be completed.*

## 4.3 Alternative for Comparison

*To be completed.*

---
# Phase 5: Evaluation

*To be completed.*

## 5.1 Technical Results

*To be completed.*

## 5.2 Cluster Profiles

*To be completed.*

## 5.3 Comparison with CSO Urban/Rural Categories

*To be completed.*

## 5.4 Comparison with Success Criteria

*To be completed.*

## 5.5 Decision: Proceed, Revise or Reject

*To be completed.*

---
# Phase 6: Deployment and Recommendations

*To be completed.*

## 6.1 Intended Use and Users

*To be completed.*

## 6.2 Monitoring and Governance

*To be completed.*

## 6.3 Future Work

*To be completed.*

---
# Personal Reflection

*To be completed at the end of the project.*

---
# Responsible AI Collaboration Statement (300–500 words)

*To be completed at the end of the project. See `AI_LOG.md` in the repository for the running record.*